# Task 4: General Health Query Chatbot
## DevelopersHub Corporation — AI/ML Engineering Internship



###  Problem Statement
Many people search online for basic health information but often land on unreliable sources. A well-designed health chatbot powered by a Large Language Model (LLM) can provide friendly, clear, and safe general health information while always reminding users to consult a real doctor for personal medical advice.

In this task, we build a General Health Query Chatbot using prompt engineering and the Hugging Face Inference API (free, no credit card required) with the `Mistral-7B-Instruct` model.

###  Goal
- Connect to a free LLM via the Hugging Face Inference API
- Use prompt engineering to shape the chatbot's personality and tone
- Implement safety filters to block harmful or dangerous queries
- Maintain multi-turn conversation history
- Test with real-world health queries

###  Tools & Libraries
| Tool | Purpose |
|------|---------|
| `requests` | Send HTTP requests to Hugging Face API |
| `python-dotenv` | Load API key securely from `.env` file |
| `Hugging Face API` | Free LLM inference (Mistral-7B-Instruct) |
| Prompt Engineering | Shape chatbot behavior and tone |

###  Model Used
- Model: `mistralai/Mistral-7B-Instruct-v0.2`
- Provider: Hugging Face Inference API (Free tier)
- Why Mistral: Strong instruction-following, great for chat, completely free to use

---
## Step 1: Import Libraries

In [1]:
# Standard Libraries 
import os
import json
import time
import textwrap

# HTTP requests to Hugging Face API 
import requests



print(' All libraries imported successfully!')

✅ All libraries imported successfully!


---
## Step 2: Load API Key Securely

> **Before running this cell:**
> 1. Create a file named `.env` in the **same folder** as this notebook
> 2. Add this single line inside it (replace with your actual token):
> ```
> HF_API_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
> ```
> 3. Save the `.env` file
>
> Get your free token at: https://console.groq.com

In [2]:
#  Groq API Setup 
from groq import Groq

# Paste your Groq API key here 
GROQ_API_KEY = "gsk_your_token_here"  # replace with your key  

# Initialize Groq client 
client = Groq(api_key=GROQ_API_KEY)

masked = GROQ_API_KEY[:8] + '****' + GROQ_API_KEY[-4:]
print(f' Groq API configured: {masked}')
print(f' Model: llama3-8b-8192 (fast & free)')

✅ Groq API configured: gsk_Snfb****vUT8
✅ Model: llama3-8b-8192 (fast & free)


---
## Step 3: Prompt Engineering

Prompt engineering means crafting a detailed instruction (called a system prompt) that tells the LLM exactly how to behave  its personality, tone, limits, and response style. This is the core skill of this task.

In [3]:
# System Prompt: defines the chatbot's personality
# carefully written instructions that shape every response
SYSTEM_PROMPT = """You are HealthBot, a friendly and knowledgeable general health information assistant.

Your role:
- Answer general health and wellness questions in a clear, simple, and friendly way
- Use plain language that anyone can understand (avoid heavy medical jargon)
- Keep responses concise  3 to 5 sentences unless more detail is clearly needed
- Always show empathy and warmth in your tone

Your strict rules:
- ALWAYS end every response with: "Remember, I'm not a doctor. Please consult a healthcare professional for personal medical advice."
- NEVER diagnose a specific illness for a specific person
- NEVER recommend specific prescription medications or exact dosages
- NEVER provide advice that could replace emergency medical care
- If someone describes a medical emergency, immediately tell them to call emergency services (115 in Pakistan, 911 in USA)

You CAN:
- Explain what symptoms commonly mean in general terms
- Describe how common medications generally work
- Share general wellness, nutrition, and lifestyle tips
- Explain medical terms in simple language
- Suggest when someone should see a doctor"""

print(' System prompt defined successfully!')
print(f'   Prompt length: {len(SYSTEM_PROMPT)} characters')
print('\n── System Prompt Preview ')
print(SYSTEM_PROMPT[:300] + '...')

✅ System prompt defined successfully!
   Prompt length: 1110 characters

── System Prompt Preview ──────────────────────────────
You are HealthBot, a friendly and knowledgeable general health information assistant.

Your role:
- Answer general health and wellness questions in a clear, simple, and friendly way
- Use plain language that anyone can understand (avoid heavy medical jargon)
- Keep responses concise — 3 to 5 sentenc...


---
## Step 4: Safety Filter

A responsible chatbot must **refuse harmful queries**. We implement a keyword-based safety filter that blocks dangerous topics before they even reach the LLM.

In [4]:
#  Safety Filter: blocked topics 
# These categories of queries will be refused immediately
# without being sent to the model
BLOCKED_KEYWORDS = [
    # Self-harm related
    'suicide', 'kill myself', 'end my life', 'self-harm',
    'cut myself', 'hurt myself',
    # Dangerous medication misuse
    'overdose', 'how many pills to', 'lethal dose',
    'drug abuse', 'get high on',
    # Illegal activities
    'illegal drugs', 'buy drugs', 'drug dealer',
]

# Emergency keywords: respond with emergency advice 
EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', 'stroke', 'cant breathe',
    "can't breathe", 'unconscious', 'not breathing',
    'severe bleeding', 'poisoning', 'overdosed'
]

def check_safety(user_input):
    """
    Checks if a user query is safe to process.

    Returns:
        ('blocked', message)   — for harmful queries
        ('emergency', message) — for medical emergencies
        ('safe', None)         — for normal queries
    """
    text = user_input.lower().strip()

    # Check for blocked keywords first
    for keyword in BLOCKED_KEYWORDS:
        if keyword in text:
            return ('blocked',
                "I'm sorry, but I'm not able to help with that topic. "
                "If you're going through a difficult time, please reach out to "
                "a mental health professional or call a helpline immediately. "
                "In Pakistan: Umang helpline 0317-4288665. "
                "You are not alone. 💙")

    # Check for emergency keywords
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in text:
            return ('emergency',
                "⚠️ This sounds like a medical emergency! "
                "Please call emergency services immediately — "
                "115 in Pakistan or 911 in the USA. "
                "Do not wait — get help right now!")

    return ('safe', None)


#  Quick test of the safety filter 
test_cases = [
    'What causes a sore throat?',   # should be safe
    'I want to hurt myself',         # should be blocked
    'I have severe chest pain',      # should be emergency
]

print(' Safety Filter Test ')
for test in test_cases:
    status, msg = check_safety(test)
    print(f'  Query  : "{test}"')
    print(f'  Status : {status.upper()}')
    if msg:
        print(f'  Response: {msg[:80]}...')
    print()

── Safety Filter Test ─────────────────────────────────
  Query  : "What causes a sore throat?"
  Status : SAFE

  Query  : "I want to hurt myself"
  Status : BLOCKED
  Response: I'm sorry, but I'm not able to help with that topic. If you're going through a d...

  Query  : "I have severe chest pain"
  Status : EMERGENCY
  Response: ⚠️ This sounds like a medical emergency! Please call emergency services immediat...



---
## Step 5: Core Chatbot Function

This function formats the conversation history into the Mistral instruction format and sends it to the Hugging Face API.

In [5]:
def ask_healthbot(user_input, conversation_history, max_retries=3):
    """
    Chatbot using Groq API with Llama3 model.
    Fast, free, no quota issues.
    """
    #  Safety check 
    status, safety_msg = check_safety(user_input)
    if status in ('blocked', 'emergency'):
        return safety_msg

    #  Add user message to history
    conversation_history.append({
        'role': 'user',
        'content': user_input
    })

    #  Build messages with system prompt 
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for msg in conversation_history:
        messages.append({
            'role'   : msg['role'] if msg['role'] != 'assistant' else 'assistant',
            'content': msg['content']
        })

    #  Send to Groq 
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model      = 'llama3-8b-8192',
                messages   = messages,
                temperature= 0.7,
                max_tokens = 300,
            )

            bot_reply = response.choices[0].message.content.strip()

            # Save to history
            conversation_history.append({
                'role'   : 'assistant',
                'content': bot_reply
            })
            return bot_reply

        except Exception as e:
            if attempt == max_retries - 1:
                return f'❌ Error: {str(e)}'
            time.sleep(3)

    return '❌ Could not get response. Please try again.'

 
print(' Groq chatbot function ready!')
print('  model = \'llama-3.3-70b-versatile\',')

print('   Safety     :  active')
print('   Multi-turn :  active')

✅ Groq chatbot function ready!
  model = 'llama-3.3-70b-versatile',
   Safety     : ✅ active
   Multi-turn : ✅ active


In [6]:
from groq import Groq
import textwrap, time

client = Groq(api_key=GROQ_API_KEY)

def ask_healthbot(user_input, conversation_history, max_retries=3):
    status, safety_msg = check_safety(user_input)
    if status in ('blocked', 'emergency'):
        return safety_msg
    conversation_history.append({'role': 'user', 'content': user_input})
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for msg in conversation_history:
        messages.append({'role': msg['role'] if msg['role'] != 'assistant' else 'assistant', 'content': msg['content']})
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model='llama-3.3-70b-versatile',
                messages=messages,
                temperature=0.7,
                max_tokens=300,
            )
            bot_reply = response.choices[0].message.content.strip()
            conversation_history.append({'role': 'assistant', 'content': bot_reply})
            return bot_reply
        except Exception as e:
            if attempt == max_retries - 1:
                return f'❌ Error: {str(e)}'
            time.sleep(3)

print(' Done! Now run Cell 6')

✅ Done! Now run Cell 6


---
## Step 6: Test the Chatbot — Example Queries

We now test the chatbot using the exact example queries from the task brief, plus additional ones.

In [7]:
def print_response(question, response):
    """Nicely formatted print of question and answer."""
    print('─' * 65)
    print(f'🧑 You    : {question}')
    print()
    # Wrap long responses for clean display
    wrapped = textwrap.fill(response, width=62,
                            initial_indent='🤖 HealthBot: ',
                            subsequent_indent='             ')
    print(wrapped)
    print()


# ── Test queries from task brief + additional 
test_queries = [
    'What causes a sore throat?',
    'Is paracetamol safe for children?',
    'What are the symptoms of diabetes?',
    'How can I improve my sleep quality?',
    'What foods are good for high blood pressure?',
]

print('=' * 65)
print('        HEALTHBOT — General Health Query Assistant')
print('=' * 65)

# Each query uses a fresh conversation (single-turn test)
for query in test_queries:
    history  = []   # fresh history for each standalone question
    response = ask_healthbot(query, history)
    print_response(query, response)
    time.sleep(2)   # small delay between requests

print('─' * 65)
print(' All test queries completed!')

       🏥 HEALTHBOT — General Health Query Assistant
─────────────────────────────────────────────────────────────────
🧑 You    : What causes a sore throat?

🤖 HealthBot: A sore throat can be really uncomfortable and
             painful. It's often caused by viral or bacterial
             infections, such as the common cold or flu, which
             can irritate the throat and make it swollen.
             Other factors like allergies, dry air, or
             shouting can also contribute to a sore throat.
             Sometimes, it can be a sign of a more serious
             issue, so if the pain persists or is severe, it's
             a good idea to see a doctor to rule out any
             underlying conditions.  Remember, I'm not a
             doctor. Please consult a healthcare professional
             for personal medical advice.

─────────────────────────────────────────────────────────────────
🧑 You    : Is paracetamol safe for children?

🤖 HealthBot: Paracetamol, also kn

---
## Step 7: Test Safety Filter

Verify that the safety system correctly handles dangerous and emergency queries.

In [8]:
#  Safety filter demonstration 
safety_tests = [
    ('SAFE',      'How much water should I drink daily?'),
    ('BLOCKED',   'How do I hurt myself?'),
    ('EMERGENCY', 'I have severe chest pain right now'),
    ('SAFE',      'What are the benefits of walking 30 minutes a day?'),
    ('BLOCKED',   'What is the lethal dose of aspirin?'),
]

print('=' * 65)
print('           SAFETY FILTER DEMONSTRATION')
print('=' * 65)

all_passed = True
for expected, query in safety_tests:
    status, msg = check_safety(query)
    passed = status.upper() == expected
    icon   = '✅' if passed else '❌'
    if not passed:
        all_passed = False

    print(f'{icon} Expected : {expected}')
    print(f'   Query    : "{query}"')
    print(f'   Result   : {status.upper()}')
    if msg:
        short_msg = msg[:90] + '...' if len(msg) > 90 else msg
        print(f'   Response : {short_msg}')
    print()

print('─' * 65)
print(f'Safety Filter Tests: {"ALL PASSED ✅" if all_passed else "SOME FAILED ❌"}')

         🛡️  SAFETY FILTER DEMONSTRATION
✅ Expected : SAFE
   Query    : "How much water should I drink daily?"
   Result   : SAFE

✅ Expected : BLOCKED
   Query    : "How do I hurt myself?"
   Result   : BLOCKED
   Response : I'm sorry, but I'm not able to help with that topic. If you're going through a difficult t...

✅ Expected : EMERGENCY
   Query    : "I have severe chest pain right now"
   Result   : EMERGENCY
   Response : ⚠️ This sounds like a medical emergency! Please call emergency services immediately — 115 ...

✅ Expected : SAFE
   Query    : "What are the benefits of walking 30 minutes a day?"
   Result   : SAFE

✅ Expected : BLOCKED
   Query    : "What is the lethal dose of aspirin?"
   Result   : BLOCKED
   Response : I'm sorry, but I'm not able to help with that topic. If you're going through a difficult t...

─────────────────────────────────────────────────────────────────
Safety Filter Tests: ALL PASSED ✅


---
## Step 8: Multi-Turn Conversation Demo

A key feature of the chatbot is **conversation memory** — it remembers previous messages and gives contextually relevant follow-up answers.

In [9]:
#  Multi-turn conversation test 
# The chatbot remembers previous messages in the conversation

print('=' * 65)
print('        MULTI-TURN CONVERSATION DEMO')
print('   (Chatbot remembers context across messages)')
print('=' * 65)

# Shared conversation history  this is what enables multi-turn chat
conversation = []

multi_turn_queries = [
    'I have been having headaches every day this week.',
    'What could be causing them?',
    'Should I be worried?',
]

for query in multi_turn_queries:
    response = ask_healthbot(query, conversation)
    print_response(query, response)
    time.sleep(2)

print(f'   Conversation turns: {len(conversation) // 2}')
print('─' * 65)

       💬 MULTI-TURN CONVERSATION DEMO
   (Chatbot remembers context across messages)
─────────────────────────────────────────────────────────────────
🧑 You    : I have been having headaches every day this week.

🤖 HealthBot: I'm so sorry to hear that you're experiencing
             daily headaches - that can be really frustrating
             and uncomfortable. There could be many reasons
             for frequent headaches, such as stress, lack of
             sleep, or dehydration. It's also possible that it
             could be related to your diet, environment, or
             even a sign of an underlying issue.   Remember,
             I'm not a doctor. Please consult a healthcare
             professional for personal medical advice.

─────────────────────────────────────────────────────────────────
🧑 You    : What could be causing them?

🤖 HealthBot: There are many potential causes of headaches, and
             it's often a combination of factors. Some common
             on

---
## Step 9: Interactive Chat (Type Your Own Questions)

Run this cell to chat with HealthBot interactively. Type `quit` to exit.

In [10]:
#  Interactive Chat Loop 
# Type your own health questions and get real responses!
# Type 'quit' or 'exit' to stop
# Type 'clear' to start a fresh conversation

print('=' * 65)
print('    HealthBot Interactive Chat')
print('   Commands: "quit" to exit | "clear" to reset conversation')
print('=' * 65)

chat_history = []   # stores the full conversation

while True:
    try:
        user_input = input('\n🧑 You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Chat ended.')
        break

    if not user_input:
        continue

    if user_input.lower() in ('quit', 'exit', 'bye'):
        print('\n HealthBot: Take care and stay healthy! Goodbye! 👋')
        break

    if user_input.lower() == 'clear':
        chat_history = []
        print('\n🔄 Conversation cleared. Starting fresh!')
        continue

    if user_input.lower() == 'history':
        print(f'\n📝 Conversation has {len(chat_history)//2} turn(s).')
        continue

    # Get response from chatbot
    print('\n⏳ HealthBot is thinking...')
    response = ask_healthbot(user_input, chat_history)

    # Display response with nice wrapping
    wrapped = textwrap.fill(response, width=60,
                            initial_indent='\n🤖 HealthBot: ',
                            subsequent_indent='             ')
    print(wrapped)

   🏥 HealthBot Interactive Chat
   Commands: "quit" to exit | "clear" to reset conversation

🤖 HealthBot: Take care and stay healthy! Goodbye! 👋


---
## Step 10: Prompt Engineering Analysis

This section explains the prompt engineering techniques used and why each one matters.

In [11]:
# Prompt Engineering Techniques Summary 
techniques = [
    {
        'technique' : 'Role Assignment',
        'used'      : 'You are HealthBot, a friendly health assistant',
        'why'       : 'Gives the model a clear persona and purpose'
    },
    {
        'technique' : 'Tone Definition',
        'used'      : 'friendly, empathetic, plain language',
        'why'       : 'Ensures responses are approachable for non-medical users'
    },
    {
        'technique' : 'Length Control',
        'used'      : '3 to 5 sentences unless more detail needed',
        'why'       : 'Prevents overly long or overly short responses'
    },
    {
        'technique' : 'Hard Rules (Negative Constraints)',
        'used'      : 'NEVER diagnose, NEVER prescribe, NEVER replace emergency care',
        'why'       : 'Critical for safety — prevents harmful medical advice'
    },
    {
        'technique' : 'Required Disclaimer',
        'used'      : 'Always end with: I am not a doctor...',
        'why'       : 'Legal and ethical protection, reminds users to seek real help'
    },
    {
        'technique' : 'Positive Permissions',
        'used'      : 'You CAN explain symptoms generally, share wellness tips...',
        'why'       : 'Defines useful scope so model is not overly restrictive'
    },
    {
        'technique' : 'Conversation History',
        'used'      : 'Full chat history passed with every request',
        'why'       : 'Enables context-aware multi-turn conversations'
    },
    {
        'technique' : 'Temperature Setting',
        'used'      : 'temperature=0.7',
        'why'       : 'Balanced creativity: not too rigid (0.0) or random (1.0)'
    },
]

print('=' * 65)
print('    PROMPT ENGINEERING TECHNIQUES USED')
print('=' * 65)
for i, t in enumerate(techniques, 1):
    print(f'\n{i}. {t["technique"]}')
    print(f'   Used    : {t["used"]}')
    print(f'   Why     : {t["why"]}')

print('\n' + '=' * 65)
print(f'Total techniques applied: {len(techniques)}')

    📐 PROMPT ENGINEERING TECHNIQUES USED

1. Role Assignment
   Used    : You are HealthBot, a friendly health assistant
   Why     : Gives the model a clear persona and purpose

2. Tone Definition
   Used    : friendly, empathetic, plain language
   Why     : Ensures responses are approachable for non-medical users

3. Length Control
   Used    : 3 to 5 sentences unless more detail needed
   Why     : Prevents overly long or overly short responses

4. Hard Rules (Negative Constraints)
   Used    : NEVER diagnose, NEVER prescribe, NEVER replace emergency care
   Why     : Critical for safety — prevents harmful medical advice

5. Required Disclaimer
   Used    : Always end with: I am not a doctor...
   Why     : Legal and ethical protection, reminds users to seek real help

6. Positive Permissions
   Used    : You CAN explain symptoms generally, share wellness tips...
   Why     : Defines useful scope so model is not overly restrictive

7. Conversation History
   Used    : Full chat his

---
## Step 11: Final Summary & Insights

###  System Architecture

```
User Input
    │
    ▼
Safety Filter  ──► BLOCKED  ──► Refuse with helpful message
    │           ──► EMERGENCY ──► Direct to emergency services
    │
    ▼ SAFE
Prompt Engineer
(System Prompt + Conversation History + User Query)
    │
    ▼
Hugging Face API
(Mistral-7B-Instruct-v0.2)
    │
    ▼
Response Cleaning & Formatting
    │
    ▼
Add to Conversation History
    │
    ▼
Display to User
```

###  Chatbot Capabilities Summary

| Feature | Status |
|---------|--------|
| General health questions | ✅ Supported |
| Multi-turn conversation | ✅ Supported |
| Safety filter (harmful) | ✅ Implemented |
| Emergency detection | ✅ Implemented |
| Prompt engineering | ✅ Applied (8 techniques) |
| API retry logic | ✅ Implemented |
| Medical diagnosis | ❌ Intentionally blocked |
| Prescription advice | ❌ Intentionally blocked |

###  Key Findings

1. **Prompt engineering is powerful**  a well-crafted system prompt can turn a general LLM into a domain-specific assistant without any model training
2. **Safety filters are essential**  they must run before the LLM, not after, to prevent harmful content generation entirely
3. **Multi-turn context** significantly improves response quality  the model gives better follow-up answers when it remembers prior messages
4. **Temperature tuning matters** 0.7 gives friendly, natural responses while keeping medical information accurate
5. **Free models work well**  Mistral-7B via Hugging Face performs comparably to GPT-3.5 for general Q&A tasks

###  Limitations
- The free Hugging Face tier has rate limits heavy usage may require waiting
- The model may occasionally hallucinate medical details; always consult a real doctor
- Keyword-based safety filters can miss novel phrasings  production systems use ML classifiers

###  Conclusion
We successfully built a **safe, friendly, and functional health information chatbot** using prompt engineering and the free Hugging Face Inference API. The system correctly handles general health questions, maintains conversation context, and enforces safety rules  demonstrating core skills in prompt design, API usage, and responsible AI development.

### Author :
Muhammad Abdullah
(DHC-8675)